# Finetune weights

In [ ]:
from pathlib import Path
import sys
import os

PROJECT_ROOT = Path.cwd().parent.resolve()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "dataset_v1"
REPORT_DIR = PROJECT_ROOT / "reports"

print("cwd =", Path.cwd())

In [ ]:
import json
import pandas as pd
import numpy as np
from pathlib import Path

import types
import importlib
import src.finetune_metric
importlib.reload(src.finetune_metric)
from src.finetune_metric import finetune

import gc
import torch


## Split data according description.json

In [ ]:
# create manifest_splits

# === Path ===
manifest_path = Path(DATA_DIR / "manifest.csv")
desc_path = Path(DATA_DIR / "description_full.json")
out_path = DATA_DIR / "manifest_with_colors.csv"

df = pd.read_csv(manifest_path)

with open(desc_path, "r", encoding="utf-8") as f:
    desc_data = json.load(f)

if isinstance(desc_data, list):
    merged = {}
    for block in desc_data:
        if isinstance(block, dict):
            merged.update(block)
    desc_data = merged

print(f"Rows in manifest: {len(df)}")
print(f"Games in simplified description: {len(desc_data)}")


# === normalize description ===
lookup = {}

for game_name, roles in desc_data.items():
    if not isinstance(roles, dict):
        continue
    for slot_name, info in roles.items():
        if not isinstance(info, dict):
            continue
        lookup[(game_name, slot_name)] = {
            "color_label": info.get("color_label"),
            "description": info.get("description"),
        }


# === Player slot determination function ===
def resolve_team_slot(row):
    label = str(row.get("label", "")).strip().lower()
    role_name = str(row.get("role_name", "")).strip().lower()

    if label == "team_left":
        return "left"
    if label == "team_right":
        return "right"

    if label == "goalkeeper_left":
        return "goalkeeper_left"
    if label == "goalkeeper_right":
        return "goalkeeper_right"

    if label in {"goalkeeper", "gk"}:
        if "left" in role_name:
            return "goalkeeper_left"
        if "right" in role_name:
            return "goalkeeper_right"
        return None

    if "goalkeeper" in role_name:
        if "left" in role_name:
            return "goalkeeper_left"
        if "right" in role_name:
            return "goalkeeper_right"
        return None

    return None


# === add team_slot ===
df["team_slot"] = df.apply(resolve_team_slot, axis=1)

# === map game + team_slot -> color_label / description ===
def map_color_label(row):
    game = row["game"]
    slot = row["team_slot"]
    if pd.isna(slot) or slot is None:
        return np.nan
    item = lookup.get((game, slot))
    if item is None:
        return np.nan
    return item.get("color_label", np.nan)

def map_description(row):
    game = row["game"]
    slot = row["team_slot"]
    if pd.isna(slot) or slot is None:
        return np.nan
    item = lookup.get((game, slot))
    if item is None:
        return np.nan
    return item.get("description", np.nan)

df["color_label"] = df.apply(map_color_label, axis=1)
df["kit_description"] = df.apply(map_description, axis=1)
df["has_color_label"] = df["color_label"].notna()

# === statistics ===
games_in_manifest = set(df["game"].dropna().unique())
games_with_desc = set(desc_data.keys())
covered_games = games_in_manifest & games_with_desc
missing_games = sorted(games_in_manifest - games_with_desc)

print("\n=== Coverage stats ===")
print(f"Unique games in manifest: {len(games_in_manifest)}")
print(f"Games with description:   {len(games_with_desc)}")
print(f"Covered games:            {len(covered_games)}")
print(f"Missing games:            {len(missing_games)}")

print("\nRows with mapped color_label:", int(df["has_color_label"].sum()))
print("Rows without mapped color_label:", int((~df["has_color_label"]).sum()))

print("\nTop color labels:")
print(df["color_label"].value_counts(dropna=False).head(20))

if missing_games:
    print("\nGames missing in description (first 20):")
    print(missing_games[:20])

# === save ===
df.to_csv(out_path, index=False, encoding="utf-8")
print(f"\nSaved: {out_path}")

display(
    df[
        [
            "crop_path",
            "label",
            "game",
            "role_name",
            "team_slot",
            "color_label",
            "kit_description",
            "has_color_label",
        ]
    ].head(20)
)

In [ ]:
MANIFEST = DATA_DIR / r"manifest_with_colors.csv"
OUT_PATH  = DATA_DIR / r"manifest_split.csv"
SEED = 42

# ── 1. Load & filter ──────────────────────────────────────────────────────────
df = pd.read_csv(MANIFEST)
df = df[df["has_color_label"] == True].copy()
print(f"Rows after filtering: {len(df)}  |  games: {df['game'].nunique()}")

# ── 2. Two dominant colors per match (team_left + team_right) ─────────────────
def top2_colors(s):
    vc = s.value_counts()
    return tuple(sorted(vc.index[:2].tolist()))

match_colors = (
    df.groupby("game")["color_label"]
    .agg(top2_colors)
    .reset_index()
    .rename(columns={"color_label": "color_pair"})
)
print(f"Total matches: {len(match_colors)}")
print(match_colors["color_pair"].value_counts())

# ── 3. Stratified split by color pair ────────────────────────────────────────
rng = np.random.default_rng(SEED)
train_matches, val_matches = [], []

for pair, group in match_colors.groupby("color_pair"):
    ids = list(group["game"])
    rng.shuffle(ids)
    n = len(ids)
    if n == 1:
        train_matches += ids
    else:
        n_val = max(1, round(n * 0.2))
        val_matches   += ids[:n_val]
        train_matches += ids[n_val:]

# ── 4. Assign split column ────────────────────────────────────────────────────
split_map = (
    {m: "train" for m in train_matches}
    | {m: "val"   for m in val_matches}
)
df["split"] = df["game"].map(split_map)

# ── 5. Stats ──────────────────────────────────────────────────────────────────
print("\nRow counts per split:")
print(df["split"].value_counts())

print("\nColor distribution per split (fraction within split):")
print(
    df.groupby("split")["color_label"]
    .value_counts(normalize=True)
    .unstack(fill_value=0)
    .round(3)
)

# ── 6. Save ───────────────────────────────────────────────────────────────────
df.to_csv(OUT_PATH, index=False)
print(f"\nSaved: {OUT_PATH}")

## Finetune models

In [ ]:
import pandas as pd

path = "dataset_v1/manifest_split.csv"
df = pd.read_csv(path)

df["crop_path"] = (
    df["crop_path"]
    .astype(str)
    .str.replace(r"^dataset_v1[\\/]", "", regex=True)
)

df.to_csv(path, index=False)

In [ ]:
manifest = "dataset_v1/manifest_split.csv"
crop_root = "dataset_v1"

In [ ]:
args_osnet_supcon = types.SimpleNamespace(
    model     = "osnet",          # "osnet" | "dino"
    loss      = "supcon",         # "supcon" | "triplet"
    manifest  = manifest,
    crop_root = crop_root,
    epochs    = 30,
    P         = 8,
    K         = 4,
    lr        = 3e-5,
    freeze_bn = True,
    ckpt_dir  = "checkpoints",
    device    = "cuda",
    resume = True
)
# ── DINO + SupCon ─────────────────────────────────────────────────────────────
args_dino_supcon = types.SimpleNamespace(
    model     = "dino",
    loss      = "supcon",
    manifest  = manifest,
    crop_root = crop_root,
    epochs    = 3,
    P         = 8,
    K         = 4,
    lr        = 1e-4,
    freeze_bn = False,  # у DINO нет BN
    ckpt_dir  = "checkpoints",
    device    = "cuda",
)

# ── DINO + Triplet ────────────────────────────────────────────────────────────
args_dino_triplet = types.SimpleNamespace(
    model     = "dino",
    loss      = "triplet",
    manifest  = manifest,
    crop_root = crop_root,
    epochs    = 3,
    P         = 8,
    K         = 4,
    lr        = 1e-4,
    freeze_bn = False,
    ckpt_dir  = "checkpoints",
    device    = "cuda",
)

# ── OSNet + Triplet ───────────────────────────────────────────────────────────
args_osnet_triplet = types.SimpleNamespace(
    model     = "osnet",          # "osnet" | "dino"
    loss      = "triplet",         # "supcon" | "triplet"
    manifest  = manifest,
    crop_root = crop_root,
    epochs    = 30,
    P         = 8,
    K         = 4,
    lr        = 3e-5,
    freeze_bn = True,
    ckpt_dir  = "checkpoints",
    device    = "cuda",
    resume = False
)

In [ ]:
for name in dir():
    if not name.startswith('_'):
        try:
            obj = eval(name)
            if hasattr(obj, 'cuda') or hasattr(obj, 'device'):
                del obj
        except:
            pass

gc.collect()
print("done")

In [ ]:
torch.cuda.empty_cache()
print(f"allocated: {torch.cuda.memory_allocated()/1e9:.3f} GB")
print(f"reserved:  {torch.cuda.memory_reserved()/1e9:.3f} GB")

In [ ]:
def free_gpu():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    print(f"GPU memory: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated")

free_gpu()

In [ ]:
history_osnet_supcon, _ = finetune(args_osnet_supcon)
free_gpu()


In [ ]:
history_dino_supcon,   _ = finetune(args_dino_supcon)
free_gpu()

In [ ]:
history_dino_triplet,  _ = finetune(args_dino_triplet)
free_gpu()

In [ ]:
history_osnet_triplet, _ = finetune(args_osnet_triplet)
free_gpu()